# Train the Arabizi → Arabic model on Colab
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

Two stages:
1. **Pre-train** a fresh model on a large synthetic set made from the Egyptian Arabic Wikipedia (downloaded here in Colab, about 200 MB, never stored in the repo).
2. **Fine-tune** it on the hand-written pairs.

Checkpoints are saved to your Google Drive every epoch. If the session is cut off, Run all again: finished stages are skipped and an interrupted one resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/arabizi'
!mkdir -p $DRIVE/runs

In [ ]:
# the code and data, from GitHub
!git clone -q https://github.com/BackendNeuron/arabizi.git /content/arabizi || (cd /content/arabizi && git pull -q)
%cd /content/arabizi
!pip -q install -r requirements.txt
# keep runs/ on Drive so checkpoints survive the session
!rm -rf runs && ln -s $DRIVE/runs runs

In [ ]:
# fine-tuning data (hand-written pairs, templates, words and chunks) and the held-out test set
!python tools/build_dataset.py --variants 4
# pre-training data from the Egyptian Arabic Wikipedia (a few minutes the first time)
!python tools/build_pretrain.py --files 2 --max-pairs 300000

In [ ]:
import os
# stage 1: pre-train (skipped once runs/pretrain-best.pt exists and the stage finished)
if not os.path.exists('runs/pretrain-done'):
    mode = '--resume' if os.path.exists('runs/pretrain-last.pt') else '--new'
    !python train.py $mode --train-file data/pretrain.tsv --name pretrain- --stage pretrain --epochs 10 --batch 256 --patience 3
    open('runs/pretrain-done', 'w').close()

In [ ]:
# stage 2: fine-tune the pre-trained model on the hand-written data, at a lower learning rate
if os.path.exists('runs/last.pt') and os.path.exists('runs/finetune-started'):
    !python train.py --resume --stage finetune --epochs 40
else:
    open('runs/finetune-started', 'w').close()
    !python train.py --continue-from runs/pretrain-best.pt --stage finetune --epochs 40

In [ ]:
!python tools/plots.py --model runs/best.pt
from IPython.display import Image, display, Markdown
for n in ['accuracy', 'cer', 'loss', 'topics']:
    display(Image(f'docs/charts/{n}-light.png'))
display(Markdown(open('docs/charts/mistakes.md', encoding='utf-8').read()))

## Later: continue training on new data
Add sentences to `data/seed/`, rebuild the dataset, then continue from the best model:
```
!python tools/build_dataset.py
!python train.py --continue-from runs/best.pt --stage more-data --epochs 20
```

**Start completely over:** delete `MyDrive/arabizi/runs` in Google Drive first.